# PCA and SVD comparison

## Comparison

Starting from the Singular Value Decomposition (SVD):

$$
\boxed{A=U\Sigma V^T}
$$

We want to demonstrate why:

$$
\boxed{AA^T=U\Sigma\Sigma^TU^T}
$$

There are only two mathematical properties we need: the transpose of a matrix product and the orthogonality of a matrix.

### 1. Calculate the transpose of A
Remember that when transposing a product of matrices, we must reverse their order:

$$
(ABC)^T=C^TB^TA^T
$$

Therefore:

$$
A^T=(U\Sigma V^T)^T
$$


$$
A^T=(V^T)^T\Sigma^TU^T
$$

Since transposing a matrix twice returns the original matrix:

$$
\boxed{A^T=V\Sigma^TU^T}
$$

## 2. Multiply A by its transpose

Substituting both expressions:

$$
AA^T=(U\Sigma V^T)(V\Sigma^TU^T)
$$

Removing the parentheses, since matrix multiplication is associative:

$$
AA^T=U\Sigma V^TV\Sigma^TU^T
$$

## 3. Apply the orthogonality property

Since $V$ is an orthogonal matrix:

$$
V^TV=VV^T=I
$$

Therefore:

$$
AA^T=U\Sigma\underbrace{V^TV}_{I}\Sigma^TU^T
$$


$$
AA^T=U\Sigma I\Sigma^TU^T
$$

And multiplying by the identity matrix does not change anything:

$$
\boxed{AA^T=U\Sigma\Sigma^TU^T}
$$

![Matrix multiplication](./images/week_4/matrix_multiplication.png)

## 4. What about the other way around?

We can follow exactly the same process for $A^TA$ instead of $AA^T$.

$$
A^TA=(V\Sigma^TU^T)(U\Sigma V^T)
$$


$$
A^TA=V\Sigma^T\underbrace{U^TU}_{I}\Sigma V^T
$$


$$
\boxed{A^TA=V\Sigma^T\Sigma V^T}
$$

Notice that this time it is $U$ that disappears due to orthogonality.

### 5. Why is this relevant to PCA?

This is where the connection between SVD and eigendecomposition becomes apparent.

Remember the general eigendecomposition of a symmetric matrix:

$$
M=Q\Lambda Q^T
$$

Comparing the three equations:

| Matrix | Eigendecomposition |
| :--- | :--- |
| General symmetric matrix | $M=Q\Lambda Q^T$ |
| $AA^T$ | $U\Sigma\Sigma^TU^T$ |
| $A^TA$ | $V\Sigma^T\Sigma V^T$ |

We can therefore conclude that:

- The columns of $U$ are the eigenvectors of $AA^T$.
- The columns of $V$ are the eigenvectors of $A^TA$.
- The squared singular values are the corresponding eigenvalues.
For example, if:

$$
\Sigma=
\begin{bmatrix}
4&0\\
0&2
\end{bmatrix}
$$

Then:

$$
\Sigma^T\Sigma=
\begin{bmatrix}
4&0\\
0&2
\end{bmatrix}
\begin{bmatrix}
4&0\\
0&2
\end{bmatrix}
=
\begin{bmatrix}
16&0\\
0&4
\end{bmatrix}
$$

## Why the covariance matrix is not required with SVD

The covariance matrix is not required with SVD because SVD can extract the principal directions and the variance along those directions directly from the centred data matrix, without having to explicitly calculate the covariance matrix.

Both approaches provide the same information, but they take different mathematical routes.


### 1. Traditional PCA using the covariance matrix

Let's assume we have a dataset $X$ with $n$ observations and $p$ features.

First, we centre the data:

$$
X_c=X-\bar{X}
$$

Then, we calculate the sample covariance matrix:

$$
C=\frac{1}{n-1}X_c^TX_c
$$

Finally, we perform eigendecomposition:

$$
\boxed{C=Q\Lambda Q^T}
$$

From this decomposition, we obtain:
- $Q$: principal directions (eigenvectors).
- $\Lambda$: variance along each principal direction (eigenvalues).

### 2. PCA using SVD

Instead of calculating the covariance matrix, we perform SVD directly on the centred data:

$$
\boxed{X_c=U\Sigma V^T}
$$

As we demonstrated earlier:

$$
X_c^TX_c=V\Sigma^T\Sigma V^T
$$

But we already know that:

$$
C=\frac{1}{n-1}X_c^TX_c
$$

Therefore, by substitution:

$$
C=\frac{1}{n-1}V\Sigma^T\Sigma V^T
$$

Rearranging:

$$
\boxed{C=V\left(\frac{\Sigma^T\Sigma}{n-1}\right)V^T}
$$

Now, compare both expressions:

$$
\begin{aligned}
C&=Q\Lambda Q^T\\[8pt]
C&=V\underbrace{\left(\frac{\Sigma^T\Sigma}{n-1}\right)}_{\Lambda}V^T
\end{aligned}
$$

We can immediately identify:

$$
\boxed{Q=V}
$$

$$
\boxed{\lambda_i=\frac{\sigma_i^2}{n-1}}
$$

Assuming matching eigenvector signs, ordering and choice of basis.

In other words, SVD already gives us all the information we wanted to extract from the covariance matrix.

### 3. Visual comparison

![PCA and SVD comparison](./images/week_4/pca_svg_comparison.png)

In [7]:
import numpy as np

rng = np.random.default_rng(7)

X = rng.multivariate_normal(
    mean=[2, 3],
    cov=[[3, 2.1], [2.1, 2]],
    size=200
)

# Centre the data
X_c = X - X.mean(axis=0)
n = X_c.shape[0]

#print(f"Matrix X:\n{X}\nMatrix X centred:\n{X_c}")

# Method 1: Covariance matrix + eigendecomposition
C = np.cov(X_c, rowvar=False, ddof=1)

eigenvalues, Q = np.linalg.eigh(C)

# Sort in descending order
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
Q = Q[:, idx]

# Method 2: SVD (without covariance matrix)
U, s, Vt = np.linalg.svd(X_c, full_matrices=False)

V = Vt.T
svd_eigenvalues = s**2 / (n - 1)

# Compare results
print("PCA eigenvalues:", eigenvalues)
print("SVD eigenvalues:", svd_eigenvalues)

print("Same eigenvalues:",
      np.allclose(eigenvalues, svd_eigenvalues))

print("Same eigenvectors (ignoring signs):",
      np.allclose(np.abs(Q), np.abs(V)))

PCA eigenvalues: [3.96364168 0.28999959]
SVD eigenvalues: [3.96364168 0.28999959]
Same eigenvalues: True
Same eigenvectors (ignoring signs): True
